# Step 3 - DPO alignment

Member A responsibility: start from the Step 1 LoRA checkpoint, train on the group's original preference pairs, and demonstrate the same-prompt behavior shift before versus after DPO.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from datasets import Dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer

candidates = [Path.cwd() / 'member A', Path.cwd().parent, Path.cwd().parent.parent / 'member A']
MEMBER_A = next(path for path in candidates if (path / 'outputs').exists())
STEP1 = MEMBER_A / 'checkpoints' / 'lora_step1'
DPO_CHECKPOINT = MEMBER_A / 'checkpoints' / 'dpo_step3'
PREFERENCE_FILE = MEMBER_A / 'preference_pairs' / 'dpo_pairs.csv'
OUTPUTS = MEMBER_A / 'outputs'
MODEL_NAME = 'distilgpt2'
if not PREFERENCE_FILE.exists():
    raise FileNotFoundError(f'Create {PREFERENCE_FILE} from dpo_pairs.template.csv and add 30 human-authored pairs.')
pairs = pd.read_csv(PREFERENCE_FILE)
required = {'prompt', 'chosen', 'rejected'}
assert required.issubset(pairs.columns), f'Missing columns: {required - set(pairs.columns)}'
assert not pairs.astype(str).apply(lambda column: column.str.contains('REPLACE_WITH').any()).any(), 'Replace all template placeholders first.'
assert len(pairs) >= 30, 'The assignment requires at least 30 pairs.'
print(pairs.head())
print('Preference rows:', len(pairs))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(STEP1)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = PeftModel.from_pretrained(base_model, STEP1, is_trainable=True)
dpo_dataset = Dataset.from_pandas(pairs[['prompt', 'chosen', 'rejected']], preserve_index=False)
dpo_config = DPOConfig(beta=0.1, max_length=256, max_prompt_length=128, per_device_train_batch_size=1, num_train_epochs=3, output_dir=str(DPO_CHECKPOINT), report_to='none', logging_steps=1)
trainer = DPOTrainer(model=model, ref_model=None, args=dpo_config, train_dataset=dpo_dataset, processing_class=tokenizer)
trainer.train()
DPO_CHECKPOINT.mkdir(parents=True, exist_ok=True)
trainer.save_model(DPO_CHECKPOINT)
tokenizer.save_pretrained(DPO_CHECKPOINT)
print('Saved DPO checkpoint:', DPO_CHECKPOINT)

In [ ]:
prompts = ['Ang opisyal nga tubag sa pangutana', 'Nagsugod na ang programa', 'Palihog ipasabut ang desisyon', 'Unsa ang angay buhaton?', 'Maayong adlaw sa tanan']
before_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL_NAME), STEP1)
after_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL_NAME), DPO_CHECKPOINT)
before_model.eval()
after_model.eval()
with (OUTPUTS / 'step3_before_after.txt').open('w', encoding='utf-8') as file:
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors='pt')
        with torch.no_grad():
            before = before_model.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
            after = after_model.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        before_text = tokenizer.decode(before[0], skip_special_tokens=True)
        after_text = tokenizer.decode(after[0], skip_special_tokens=True)
        file.write(f'PROMPT: {prompt}\nSTEP1: {before_text}\nDPO: {after_text}\n' + '-' * 80 + '\n')
print('Saved comparison:', OUTPUTS / 'step3_before_after.txt')